<a href="https://colab.research.google.com/github/bei931016/MachineLearning/blob/main/0704_Colab_LINE_Bot_with_GEMINI_Tooluse%20copy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

In [19]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051

In [20]:
import os
from pyngrok import ngrok

In [21]:
ngrok.kill()

In [22]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://extinct-perkiness-shrapnel.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://extinct-perkiness-shrapnel.ngrok-free.dev


True

In [23]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

google_search_tool = Tool(
   google_search=GoogleSearch()
)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        system_instruction="你是一個中文的AI助手，請用繁體中文回答",
        tools=[google_search_tool],
        response_modalities=["TEXT"],
    )
)

In [24]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [25]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學（Minghsin University of Science and Technology，簡稱明新科大）是一所位於台灣新竹縣新豐鄉的私立科技大學。學校地理位置優越，地處新竹科學園區、新竹工業區及竹北生醫園區的核心點，交通便利，毗鄰工業區、科學園區，享有豐富的產學合作資源。

學校沿革：
明新科技大學創立於1966年，當時名為「明新工業專科學校」，設有機械、土木、工業管理三科。 隨著時代演進，學校不斷擴展科系，並於1997年改制為「明新技术學院」。 最終在2002年9月，奉教育部核准升格為「明新科技大學」。

辦學理念與願景：
明新科大的校名「明新」取自《大學》「在明明德，在新民，在止於至善」的精義，旨在闡揚人類固有的德性與情操，期盼學子涵養高尚品德，擁有專業學問與優良技術。 學校以「深耕在地、放眼國際」為願景，目標是培養具備實務經驗與人文素養的專業人才。 在劉國偉校長的帶領下，明新科大明確定位為「一流產業大學」，致力於培養跨域整合、務實創新與全人學習的專業人才。 其校訓為「堅毅、求新、創造」。

學術單位：
明新科技大學現設有六個學院，包括：半導體學院、工程學院、管理學院、民生學院、人文與設計學院以及共同教育學院。 其中，工學院所屬各系均已通過IEET中華工程教育學會的認證。 學校在2022年更獲教育部核准通過「半導體科技博士學位學程」，這是該校成立56年來的第一個博士班，顯示其在半導體領域的深耕與發展。

學校特色：
*   **與產業緊密結合**：明新科大建校57年來，始終掌握世界與產業趨勢，不斷創新優化學校方針。 學校與產業界建立多項產學合作計畫，尤其在半導體產業方面，更是業界最愛聘用的畢業生之一，與台灣頂尖大學齊名。
*   **卓越的半導體人才培育**：學校的半導體學院不僅地理位置優越，擁有豐富的產學合作資源，更積極跨足國際，計畫與美國、日本、馬來西亞、澳洲及越南等國家的學術單位合作半導體國際人才培育計畫。
*   **「MUST」四大育才特色**：為鎖定半導體、AI、元宇宙、風電綠能等前瞻產業，明新科技大學發展出MUST四大育才特色，包括多元學習（Multidisciplinary Learning）、全球視野（Universal Perspective）、永續經營（Sustainable Operations）與技術創新（Technol

In [26]:
result2 = stateful_query("校長是誰？")
print(result2)

None


In [ ]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        if text.startswith('AI '):
            prompt = text[3:]
            reply_text = stateful_query(prompt)
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 20:31:45] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"U65a5903fead2a718b0605fd2f000cc2d","events":[]}
BODY:  {"destination":"U65a5903fead2a718b0605fd2f000cc2d","events":[{"type":"message","message":{"type":"text","id":"617022128235479211","quoteToken":"ZbLjRFKDlKjYWDR4ujQ6eSnMsyK4V2Fz60c0d-qqlxZX1raraQuTzhXervMJePIXWsBtoPE0Y6dQ0iIIeUYpZvByBEXRfovRbO5_Uq09rBDYvT8shi2-MC7cHiKhevJAB4wNb3Xsad7nZ7ldIDs3mg","markAsReadToken":"i53106NMfBIjGENx7Rk59qZY8VvOCiRBI9kry1Sv2IBkjMJd5GRqsG5R3rNoWUtcKZVrmORQyrb0UnFHAjC-YpZmOxNk1pBntl-DIT00UoMBEGkFbfZuI5NudW_bjowRaguFCHN7D17iD32kgs78AZKEfq07bGJoiiG8BKlbVz0rmWY4CPOZ52oTyN9xpYmftCoLRAlPhIK7SBtuE-Kg8g","text":"AI 簡介明新科技大學30字以內"},"webhookEventId":"01KTA5CTZEJZYEGS9J8RN3CAQ5","deliveryContext":{"isRedelivery":false},"timestamp":1780605151731,"source":{"type":"user","userId":"U91f814f19b2a063b0b6709c4e9754128"},"replyToken":"de8085fb60f74ce3ab805b2670a5820e","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 20:32:34] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"U65a5903fead2a718b0605fd2f000cc2d","events":[{"type":"message","message":{"type":"text","id":"617022145784447215","quoteToken":"1gNjqDaIEz8B6n2OUfyRQS-KnU6mkH4dfX--ptzkmL9W2c_DDznfwB9KPTivQHATVwWbzk1OD-5zhI9tBI0bhSf0OLJJjZTCKe_Wt88dKKWz75nNegaMnrYN8uIJolZcA6_MAYFlZ42-mINlP8EERg","markAsReadToken":"eXLtPjHyHHwHXqUGZXP4QjglALTepnrezb_UPQdIlYRoBmxLvMp0z3XJsgCVd4MD6Qfs3Obv0Hvc9vh7LqJXCOP92ICaPeDzzRsLZTGm77TNTwFRchHy7ABYUDj9DNkalAUPOX_1OQMNBVuJlZSIs6T2fE4DgCfO0dVqDYBQh-50oeDbr7ZRIJzYl7PN6DxAv9GnqUgeBGdG9KHbM67B3w","text":"AI 校長是誰"},"webhookEventId":"01KTA5D56D2WJHAW11YYQVCHFG","deliveryContext":{"isRedelivery":false},"timestamp":1780605162193,"source":{"type":"user","userId":"U91f814f19b2a063b0b6709c4e9754128"},"replyToken":"34478c40c8964c17be7e194eb1d14aa4","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 20:32:45] "POST / HTTP/1.1" 200 -
